In [13]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
pedestrians_in_traffic = fetch_ucirepo(id=536) 
  
# data (as pandas dataframes) 
X = pedestrians_in_traffic.data.features 
y = pedestrians_in_traffic.data.targets 

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
df = pd.concat([X, y], axis=1)

In [16]:
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values per column:
body_roll     2698
body_pitch    2698
body_yaw      2698
head_roll     2698
head_pitch    2698
head_yaw      2698
dtype: int64


In [17]:
# Handle missing values by imputing Mode
for col in ['body_roll', 'body_pitch', 'body_yaw','head_roll','head_pitch','head_yaw']:
    df[col] = df[col].fillna(df[col].mode()[0])

print('\nRemaining missing values:', df.isnull().sum().sum())


Remaining missing values: 0


In [18]:
# Count duplicate records
print("Duplicate records:", df.duplicated().sum())

# Remove duplicate records
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)
print("Duplicates remaining:", df.duplicated().sum())

Duplicate records: 0
Shape after removing duplicates: (4759, 14)
Duplicates remaining: 0


In [19]:
# Select numerical columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Calculate number of outliers in each numerical column
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_limit) |
        (df[col] > upper_limit)
    ]

    print(f"{col}: {len(outliers)} outliers")

oid: 0 outliers
timestamp: 0 outliers
x: 0 outliers
y: 0 outliers
body_roll: 411 outliers
body_pitch: 481 outliers
body_yaw: 2049 outliers
head_roll: 831 outliers
head_pitch: 1052 outliers
head_yaw: 1681 outliers


In [20]:
Q1 = df['body_roll'].quantile(0.25)
Q3 = df['body_roll'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['body_roll'] < lower_bound) | (df['body_roll'] > upper_bound)]
print(f'IQR: {IQR}, Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')
print('Outliers count in hours-per-week:', len(outliers))

# Treat outliers using Capping (Winsorization)
df['body_roll'] = df['body_roll'].clip(lower=lower_bound, upper=upper_bound)
print('Outliers treated! Capped Min:', df['body_roll'].min(), 'Capped Max:', df['body_roll'].max())

IQR: 0.0, Lower Bound: 0.0, Upper Bound: 0.0
Outliers count in hours-per-week: 411
Outliers treated! Capped Min: 0.0 Capped Max: 0.0
